In [ ]:
#1 ── Ячейка 1: Импорты ─────────────────────────────────────────
import os, logging, importlib.util
import numpy as np
import pandas as pd
import databento as db

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S')
log = logging.getLogger('amt')

PATH_NQ_ES = './data/GLBX-20260603-XBMUUNEJJQ 16yr nq es data/glbx-mdp3-20100606-20260602.ohlcv-1m.dbn'
CACHE_DIR  = './cache'
CLASSIFY   = './amt_classify.py'
os.makedirs(CACHE_DIR, exist_ok=True)
log.info('OK')

In [ ]:
#2 ── VIX загрузка ──────────────────────────────────────────
import databento as db

VIX_CACHE = os.path.join(CACHE_DIR, 'vix_daily.parquet')

if os.path.exists(VIX_CACHE):
    vix = pd.read_parquet(VIX_CACHE)
    log.info('VIX загружен из кеша: %d дней', len(vix))
else:
    from fredapi import Fred
    fred = Fred(api_key='YOUR_FRED_API_KEY')
    vix_series = fred.get_series('VIXCLS', 
                                  observation_start='2010-01-01',
                                  observation_end='2026-06-01')
    vix = vix_series.to_frame('vix_close')
    vix.index = pd.to_datetime(vix.index)
    vix = vix.dropna()
    
    def vix_regime(v):
        if v < 15:   return 'low'
        if v < 25:   return 'normal'
        return 'high'
    
    vix['vix_regime'] = vix['vix_close'].apply(vix_regime)
    vix.to_parquet(VIX_CACHE)
    log.info('VIX построен через FRED: %d дней', len(vix))

log.info('VIX режимы:\n%s', vix['vix_regime'].value_counts().to_string())

# Три режима
def vix_regime(v):
    if v < 15:   return 'low'
    if v < 25:   return 'normal'
    return 'high'

vix['vix_regime'] = vix['vix_close'].apply(vix_regime)
log.info('VIX режимы:\n%s', vix['vix_regime'].value_counts().to_string())

In [ ]:
#3 ── Макро данные: ZT, ZN, CL, DX ─────────────────────────
MACRO_PATHS = {
    'zn': './data/GLBX-20260604-YY86KWTSMD 10yr/glbx-mdp3-20100606-20260605.ohlcv-1d.dbn',
    'zt': './data/GLBX-20260606-BTSF5RNAY9 zt 2yr/glbx-mdp3-20100606-20260605.ohlcv-1d.dbn',
    'cl': './data/GLBX-20260606-EWECGEANF7 cl/glbx-mdp3-20100606-20260605.ohlcv-1d.dbn',
    'dx': './data/IFUS-20260606-3B38JP5FG8 2 dx 2018/ifus-impact-20181223-20260605.ohlcv-1d.dbn',
}

def load_macro(name, path, symbol_filter=None, cache_dir=CACHE_DIR):
    cache = os.path.join(cache_dir, f'{name}_daily.parquet')
    if os.path.exists(cache):
        df = pd.read_parquet(cache)
        log.info('%s загружен из кеша: %d дней', name.upper(), len(df))
        return df
    store = db.DBNStore.from_file(path)
    df    = store.to_df()
    df.index = pd.to_datetime(df.index).tz_localize(None)

    # Фильтруем только чистые контракты без спредов
    if symbol_filter:
        df = df[df['symbol'].str.startswith(symbol_filter) &
                ~df['symbol'].str.contains('-')].copy()

    df['date'] = df.index.date
    if 'symbol' in df.columns and df['symbol'].nunique() > 1:
        daily_vol   = df.groupby(['date', 'symbol'])['volume'].sum()
        front       = daily_vol.groupby('date').idxmax().apply(lambda x: x[1])
        df['front'] = df['date'].map(front)
        df          = df[df['symbol'] == df['front']]

    daily = df.groupby('date').agg(
        open=('open', 'first'), high=('high', 'max'),
        low=('low', 'min'),     close=('close', 'last'),
        volume=('volume', 'sum')
    )
    daily.index = pd.to_datetime(daily.index)
    daily.to_parquet(cache)
    log.info('%s построен: %d дней', name.upper(), len(daily))
    return daily

zn = load_macro('zn', MACRO_PATHS['zn'], symbol_filter='ZN')
zt = load_macro('zt', MACRO_PATHS['zt'],        symbol_filter='ZT')
cl = load_macro('cl', MACRO_PATHS['cl'],        symbol_filter='CL')
dx = load_macro('dx', MACRO_PATHS['dx'])

# DX до 2018 добиваем через FRED
DX_FRED_CACHE = os.path.join(CACHE_DIR, 'dx_fred.parquet')
if not os.path.exists(DX_FRED_CACHE):
    from fredapi import Fred
    fred     = Fred(api_key='YOUR_FRED_API_KEY')
    dx_fred  = fred.get_series('DTWEXBGS', observation_start='2010-01-01',
                                observation_end='2018-12-22').to_frame('close')
    dx_fred.index = pd.to_datetime(dx_fred.index)
    dx_fred.to_parquet(DX_FRED_CACHE)
    log.info('DX FRED загружен: %d дней', len(dx_fred))
else:
    dx_fred = pd.read_parquet(DX_FRED_CACHE)

# Объединяем DX: FRED 2010-2018 + Databento 2018-2026
dx_full = pd.concat([
    dx_fred[['close']].rename(columns={'close': 'close'}),
    dx[['close']]
]).sort_index()
dx_full = dx_full[~dx_full.index.duplicated(keep='last')]

# Yield curve slope: ZN 10Y - ZT 2Y (в базисных пунктах)
zn_close = zn['close'].rename('zn_close')
zt_close = zt['close'].rename('zt_close')
cl_close = cl['close'].rename('cl_close')
dx_close = dx_full['close'].rename('dx_close')

macro = pd.concat([zn_close, zt_close, cl_close, dx_close], axis=1)
macro['yield_slope'] = macro['zn_close'] - macro['zt_close']
macro['cl_ret']      = macro['cl_close'].pct_change()
macro['dx_ret']      = macro['dx_close'].pct_change()

# Режимы
macro['yield_regime'] = pd.cut(macro['yield_slope'],
    bins=[-999, 0, 1.0, 999],
    labels=['inverted', 'flat', 'normal'])

macro['cl_regime'] = pd.cut(macro['cl_close'],
    bins=[0, 50, 80, 999],
    labels=['low', 'normal', 'high'])

macro['dx_regime'] = pd.cut(macro['dx_close'],
    bins=[0, 95, 105, 999],
    labels=['weak', 'neutral', 'strong'])

log.info('Macro готов: %d дней\n%s',
    len(macro),
    macro[['yield_slope','yield_regime','cl_regime','dx_regime']].tail(5).to_string())

In [ ]:
#4 ── Ячейка 2: Загрузка данных (с кешем) ───────────────────────
ES_CONT = os.path.join(CACHE_DIR, 'es_continuous.parquet')
NQ_CONT = os.path.join(CACHE_DIR, 'nq_continuous.parquet')

if os.path.exists(ES_CONT) and os.path.exists(NQ_CONT):
    es_continuous = pd.read_parquet(ES_CONT)
    nq_continuous = pd.read_parquet(NQ_CONT)
    log.info('Continuous загружены из кеша  ES %s  NQ %s баров',
             f'{len(es_continuous):,}', f'{len(nq_continuous):,}')
else:
    store = db.DBNStore.from_file(PATH_NQ_ES)
    df    = store.to_df()
    df.index = df.index.tz_localize(None)
    log.info('DBN загружен: %s строк', f'{len(df):,}')

    front_symbols = [s for s in df['symbol'].unique() if '-' not in s]
    df_clean = df[df['symbol'].isin(front_symbols)].copy()
    es = df_clean[df_clean['symbol'].str.startswith('ES')].copy()
    nq = df_clean[df_clean['symbol'].str.startswith('NQ')].copy()

    def get_front_contract(data):
        data = data.copy()
        data['date'] = data.index.date
        daily_vol    = data.groupby(['date', 'symbol'])['volume'].sum()
        return daily_vol.groupby('date').idxmax().apply(lambda x: x[1])

    def build_continuous(data, front_contracts):
        data          = data.copy()
        data['date']  = data.index.date
        data['front'] = data['date'].map(front_contracts)
        return data[data['symbol'] == data['front']].drop(
            columns=['date', 'front']).sort_index()

    es_continuous = build_continuous(es, get_front_contract(es))
    nq_continuous = build_continuous(nq, get_front_contract(nq))
    es_continuous.to_parquet(ES_CONT)
    nq_continuous.to_parquet(NQ_CONT)
    log.info('Continuous сохранены в кеш')

log.info('ES %s → %s  NQ %s → %s',
         es_continuous.index[0].date(), es_continuous.index[-1].date(),
         nq_continuous.index[0].date(), nq_continuous.index[-1].date())

In [ ]:
#5 ── Ячейка 3: RTH фильтр — DST-AWARE (09:30-16:00 America/New_York) ──────
# БЫЛО: фиксированные UTC-минуты 14:30-21:00. Это сессия США только зимой (EST).
# Летом (EDT) реальная сессия 13:30-20:00 UTC, а старый фильтр брал 10:30-17:00 ET:
#   open  = цена 10:30 (не открытие), IB = ВТОРОЙ час, + лишний час после закрытия.
# 65% дней истории — летние, т.е. считались по неправильному окну (молча: баров всё
# равно ~390). Единый источник правды теперь session.py.
import sys, importlib
sys.path.insert(0, '..')
import session; importlib.reload(session)

es_rth = session.get_rth(es_continuous)
nq_rth = session.get_rth(nq_continuous)
log.info('ES RTH: %s баров / %d сессий   NQ RTH: %s баров / %d сессий',
         f'{len(es_rth):,}', es_rth.index.normalize().nunique(),
         f'{len(nq_rth):,}', nq_rth.index.normalize().nunique())


In [ ]:
#6 ── Ячейка 4: Загрузка amt_classify ───────────────────────────
spec = importlib.util.spec_from_file_location('amt_classify', CLASSIFY)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
log.info('amt_classify загружен')

In [ ]:
#7 ── Ячейка 5: Value Area база (с кешем) ───────────────────────
ES_VA = os.path.join(CACHE_DIR, 'es_va.parquet')
NQ_VA = os.path.join(CACHE_DIR, 'nq_va.parquet')

if os.path.exists(ES_VA) and os.path.exists(NQ_VA):
    es_va = pd.read_parquet(ES_VA)
    nq_va = pd.read_parquet(NQ_VA)
    log.info('VA база загружена из кеша  ES %d  NQ %d дней',
             len(es_va), len(nq_va))
else:
    es_va = mod.build_va_database(es_rth)
    nq_va = mod.build_va_database(nq_rth)
    for va, path in [(es_va, ES_VA), (nq_va, NQ_VA)]:
        save = va.copy()
        save['hvn_levels'] = save['hvn_levels'].apply(str)
        save['lvn_levels'] = save['lvn_levels'].apply(str)
        save.to_parquet(path)
    log.info('VA база построена  ES %d  NQ %d дней',
             len(es_va), len(nq_va))

log.info('Пример ES VA:\n%s',
         es_va[['poc','vah','val','day_high','day_low']].head(3).to_string())

In [ ]:
#8
def prep_va(va_db):
    va = va_db.copy()
    for col in ['hvn_levels', 'lvn_levels']:
        if col not in va.columns:
            va[col] = [[] for _ in range(len(va))]
            continue
        va[col] = va[col].apply(
            lambda x: x if isinstance(x, list) else
            [] if str(x) in ('[]', 'nan', '') else
            [float(v) for v in str(x).strip('[]').split(',') if v.strip()])
    return va

In [ ]:
#9 Считаем intraday_rv заранее и передаём в classify_days
irv_es = mod.calc_intraday_rv(es_rth, window=20)
irv_nq = mod.calc_intraday_rv(nq_rth, window=20)

# Строим словарь date -> irv для быстрого доступа
irv_es_dict = {d.date(): v for d, v in irv_es.items() if not np.isnan(v)}
irv_nq_dict = {d.date(): v for d, v in irv_nq.items() if not np.isnan(v)}
log.info('IRV ES: %.4f → %.4f', min(irv_es_dict.values()), max(irv_es_dict.values()))

# Затем classify
es_days = mod.classify_days(es_rth, prep_va(es_va), irv_dict=irv_es_dict)
nq_days = mod.classify_days(nq_rth, prep_va(nq_va), irv_dict=irv_nq_dict)



In [ ]:
#10 ── Ячейка 6: Классификация дней (с кешем) ────────────────────
ES_DAYS = os.path.join(CACHE_DIR, 'es_days.parquet')
NQ_DAYS = os.path.join(CACHE_DIR, 'nq_days.parquet')

def prep_va(va_db):
    """Восстанавливает hvn/lvn списки из parquet (строки или уже списки)."""
    va = va_db.copy()
    for col in ['hvn_levels', 'lvn_levels']:
        if col not in va.columns:
            va[col] = [[] for _ in range(len(va))]
            continue
        va[col] = va[col].apply(
            lambda x: x if isinstance(x, list) else
            [] if str(x) in ('[]', 'nan', '') else
            [float(v) for v in str(x).strip('[]').split(',') if v.strip()])
    return va

REQUIRED_COLS = {'bimodality'}  # маркеры свежего классификатора
_cache_ok = os.path.exists(ES_DAYS) and os.path.exists(NQ_DAYS)
if _cache_ok:
    es_days = pd.read_parquet(ES_DAYS)
    nq_days = pd.read_parquet(NQ_DAYS)
    _missing = REQUIRED_COLS - set(es_days.columns)
    if _missing:
        _cache_ok = False
        log.info('Кеш устарел (нет %s) — пересчёт классификации', _missing)

if _cache_ok:
    log.info('Days загружены из кеша  ES %d  NQ %d', len(es_days), len(nq_days))
else:
    es_days = mod.classify_days(es_rth, prep_va(es_va), irv_dict=irv_es_dict)
    nq_days = mod.classify_days(nq_rth, prep_va(nq_va), irv_dict=irv_nq_dict)
    es_days.to_parquet(ES_DAYS)
    nq_days.to_parquet(NQ_DAYS)
    log.info('Days построены  ES %d  NQ %d', len(es_days), len(nq_days))

log.info('Колонки: %s', list(es_days.columns))

In [ ]:
#11 ── Ячейка 7: RV-20 + Volume Z-score + IB/RV ratio ────────────
def add_normalizations(days_df, rth_data):
    rth         = rth_data.copy()
    rth['date'] = rth.index.date
    daily_close = rth.groupby('date')['close'].last()
    daily_vol   = rth.groupby('date')['volume'].sum()
    daily_close.index = pd.to_datetime(daily_close.index)
    daily_vol.index   = pd.to_datetime(daily_vol.index)

    log_ret  = np.log(daily_close / daily_close.shift(1))
    rv20     = log_ret.rolling(20).std() * np.sqrt(252)
    vol_mean = daily_vol.rolling(60).mean()
    vol_std  = daily_vol.rolling(60).std()
    vol_z    = (daily_vol - vol_mean) / vol_std.replace(0, np.nan)

    df = days_df.copy()
    df.index = pd.to_datetime(df.index)
    df['rv20']        = rv20.reindex(df.index)
    df['vol_zscore']  = vol_z.reindex(df.index)
    df['ib_pct']      = df['ib_range'] / df['open_px']
    df['ib_rv_ratio'] = df['ib_pct'] / (df['rv20'] / np.sqrt(252))
    df['ib_size']     = pd.qcut(
        df['ib_rv_ratio'].dropna(),
        q=3, labels=['narrow', 'normal', 'wide'], duplicates='drop')
    return df

es_days = add_normalizations(es_days, es_rth)
nq_days = add_normalizations(nq_days, nq_rth)

log.info('ES  RV20: %.3f → %.3f  Vol-Z: %.2f → %.2f',
         es_days['rv20'].min(), es_days['rv20'].max(),
         es_days['vol_zscore'].min(), es_days['vol_zscore'].max())

In [ ]:
#12 ── Ячейка 8: Сохранение финального датасета ──────────────────
es_days.to_parquet(ES_DAYS)
nq_days.to_parquet(NQ_DAYS)
log.info('Сохранено: es_days  nq_days  (%d колонок)', len(es_days.columns))

In [ ]:
#13 ── Ячейка 9: Статистика ──────────────────────────────────────
df = es_days.dropna(subset=['rv20']).copy()

log.info('Условия открытия:\n%s',
    df['open_location'].value_counts(normalize=True)
      .mul(100).round(1).to_string())

log.info('Типы дня:\n%s',
    df['day_type'].value_counts(normalize=True)
      .mul(100).round(1).to_string())

log.info('Типы открытия:\n%s',
    df['open_type'].value_counts(normalize=True)
      .mul(100).round(1).to_string())

log.info('Activity:\n%s',
    df['activity'].value_counts(normalize=True)
      .mul(100).round(1).to_string())

log.info('Тип дня по условию открытия:\n%s',
    df.groupby('open_location')['day_type']
      .value_counts(normalize=True).mul(100).round(1)
      .unstack(fill_value=0).to_string())

log.info('Activity по условию открытия:\n%s',
    df.groupby('open_location')['activity']
      .value_counts(normalize=True).mul(100).round(1)
      .unstack(fill_value=0).to_string())

df['direction_correct'] = (
    ((df['direction'] == 'long')  & (df['close_px'] > df['open_px'])) |
    ((df['direction'] == 'short') & (df['close_px'] < df['open_px']))
)
log.info('Win rate направления:\n%s',
    df.groupby('open_location')['direction_correct']
      .agg(win_rate=lambda x: round(x.mean()*100, 1), count='count')
      .to_string())

log.info('POC acceptance rate:\n%s',
    df.groupby('open_location')['poc_accepted']
      .agg(acceptance_rate=lambda x: round(x.mean()*100, 1), count='count')
      .to_string())

log.info('IB size distribution:\n%s',
    df['ib_size'].value_counts().to_string())

log.info('Тип дня по IB size:\n%s',
    df.groupby('ib_size')['day_type']
      .value_counts(normalize=True).mul(100).round(1)
      .unstack(fill_value=0).to_string())

In [ ]:
#14 Посмотреть реальные числа для нескольких известных дней
sample = es_days[['ib_range', 'day_range', 'ib_profile_ratio',
                   'ib_width_cat', 'profile_width_cat', 
                   'ib_profile_cat', 'day_type']].head(20)
print(sample.to_string())

# Распределение ib_profile_ratio
print(es_days['ib_profile_ratio'].describe())
print(es_days['ib_profile_ratio'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · L0 — Causal stationary feature set (regime engine)
# ═══════════════════════════════════════════════════════════════════════
# Анализ старого #17: сырые уровни cl_close/dx_close/vix_close нестационарны
# (кластеры кодируют эпоху, а не состояние рынка); day_of_week/month как
# непрерывные фичи некорректны; er_10d/trend_r2 считались зря; best_n=13 был
# захардкожен мимо BIC. Здесь — только стационарные, декоррелированные фичи.

# Стационарные замены уровням (импульсы вместо цен)
es_days['cl_mom_20d'] = macro['cl_close'].pct_change(20, fill_method=None).reindex(es_days.index)
es_days['dx_mom_20d'] = macro['dx_close'].pct_change(20, fill_method=None).reindex(es_days.index)

REGIME_FEATURES = [
    # — Market profile (нормированные на IRV, стационарные)
    'ib_norm', 'prof_norm', 'ib_profile_ratio', 'poc_bias', 'open_poc_dist',
    # — Волатильность (относительная, не абсолютный уровень)
    'vix_percentile_252d', 'vix_change_5d', 'vol_ratio_5_20',
    # — Ставки / интермаркет (спред, изменение, импульсы)
    'yield_slope', 'yield_slope_change_10d',
    'zn_mom_10d', 'cl_mom_20d', 'dx_mom_20d',
    # — Структура: тренд vs mean-reversion
    'er_20d', 'autocorr_lag1_20d',
    # — Кросс-корреляции с интермаркетом
    'cl_es_corr_20d', 'zn_es_corr_20d',
    # — Распределение доходностей + диапазон
    'ret_skew_20d', 'range_pct_20d',
]
# Исключены vs старый набор: cl_close, dx_close, vix_close (уровни),
# day_of_week, month (календарь), zn_mom_20d/es_nq_corr/ret_kurt/hurst/
# trend_r2/gk_vol_20d/vol_of_vol (дубли/шум). Список — правится под тезис.

_complete = es_days[REGIME_FEATURES].dropna()
log.info('L0: %d признаков, полных строк %d из %d (%.0f%%)',
         len(REGIME_FEATURES), len(_complete), len(es_days),
         100*len(_complete)/len(es_days))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · L1 — Regime engine (expanding, без lookahead)
# ═══════════════════════════════════════════════════════════════════════
# Чистые функции: fit на train-окне → predict на test; каноникализация
# компонент к эталонным центроидам (Hungarian), чтобы «смысл» режима был
# стабилен между окнами. diag-ковариация = меньше параметров, устойчивее.
from scipy.optimize import linear_sum_assignment
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler

N_REGIMES     = 7            # было 13; стабильно на expanding-окне с diag-cov
COV_TYPE      = 'diag'
DISCOVERY_END = '2019-01-01' # граница discovery/holdout

def fit_regime_model(train_df, features=REGIME_FEATURES, k=N_REGIMES, seed=42, n_init=10):
    """RobustScaler + GMM на train-окне (только прошлое)."""
    X = train_df[features].dropna()
    scaler = RobustScaler().fit(X)
    gmm = GaussianMixture(n_components=k, covariance_type=COV_TYPE,
                          random_state=seed, n_init=n_init, reg_covar=1e-6)
    gmm.fit(scaler.transform(X))
    return scaler, gmm

def _centroids(scaler, gmm):
    """Центроиды компонент обратно в исходном фич-пространстве (для матчинга)."""
    return scaler.inverse_transform(gmm.means_)

def build_reference(discovery_df, features=REGIME_FEATURES, k=N_REGIMES):
    """Эталонные центроиды режимов — фиксируем смысл один раз на discovery."""
    scaler, gmm = fit_regime_model(discovery_df, features, k)
    return _centroids(scaler, gmm)

def canonical_map(scaler, gmm, ref_centroids):
    """comp_id -> canonical regime: венгерский матчинг по расстоянию центроидов."""
    C   = _centroids(scaler, gmm)
    std = ref_centroids.std(axis=0) + 1e-9
    D   = np.linalg.norm((C[:, None, :] - ref_centroids[None, :, :]) / std, axis=2)
    rows, cols = linear_sum_assignment(D)
    return {int(r): int(c) for r, c in zip(rows, cols)}

def assign_regimes(df, scaler, gmm, cmap, features=REGIME_FEATURES):
    """Возвращает (regime, confidence) по строкам df с полными фичами."""
    X     = df[features]
    valid = X.dropna().index
    Z     = scaler.transform(X.loc[valid])
    comp  = gmm.predict(Z)
    regime = pd.Series(np.nan, index=df.index, name='regime')
    conf   = pd.Series(np.nan, index=df.index, name='regime_confidence')
    regime.loc[valid] = [cmap[int(c)] for c in comp]
    conf.loc[valid]   = gmm.predict_proba(Z).max(axis=1)
    return regime, conf

# Эталонные центроиды на discovery (<2019) — единое каноническое label-space
REF_CENTROIDS = build_reference(es_days[es_days.index < DISCOVERY_END])
log.info('L1: эталон %d режимов на discovery (<%s), cov=%s',
         N_REGIMES, DISCOVERY_END, COV_TYPE)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · L1b — Session-shape слой (форма завершённой сессии)
# ═══════════════════════════════════════════════════════════════════════
# Заменяет хэндмейд day_type/fuzzy: форма дня m открывается через GMM-кластер.
# Все фичи — на ЗАВЕРШЁННОМ дне m (при next-day входе lookahead нет), лаг на m+1
# делается в L2. Тот же движок L1 (expanding fit + каноникализация).

def causal_pctrank(s, w=30):
    """Перцентиль-ранг последнего значения в трейлинг-окне (causal, [0..1])."""
    return s.rolling(w).apply(lambda a: (a[-1] >= a).mean(), raw=True)

# x1,x2 — causal 30d перцентиль ширины IB и профиля (нормированных на IRV)
es_days['x1_ibw'] = causal_pctrank(es_days['ib_norm'],   30)
es_days['x2_prw'] = causal_pctrank(es_days['prof_norm'], 30)
# x5 — позиция POC в диапазоне дня (0=low, 1=high)
_rng = (es_days['day_high'] - es_days['day_low']).replace(0, np.nan)
es_days['poc_position'] = ((es_days['curr_poc'] - es_days['day_low']) / _rng).clip(0, 1)
# signed conviction: смещение close от open в единицах ожидаемого дневного хода
es_days['session_thrust'] = (es_days['close_px'] - es_days['open_px']) / (es_days['irv'] * es_days['open_px'])

SHAPE_FEATURES = [
    'x1_ibw',            # ширина IB (30d causal перцентиль)
    'x2_prw',            # ширина профиля (30d causal перцентиль)
    'ib_profile_ratio',  # x3: IB / day range
    'open_poc_dist',     # x4: |open - POC| / range
    'poc_position',      # x5: положение POC в диапазоне
    'bimodality',        # x6: бимодальность профиля (из amt_classify)
    'session_thrust',    # signed conviction (close-open)/(irv*open)
]
N_SHAPES = 6   # discovered формы сессии (вместо 6 хэндмейд day_type)

_sh = es_days[SHAPE_FEATURES].dropna()
log.info('L1b: %d shape-фич, полных строк %d из %d (%.0f%%)',
         len(SHAPE_FEATURES), len(_sh), len(es_days), 100*len(_sh)/len(es_days))

# Эталонные центроиды форм на discovery (<2019) — стабильное label-space форм
REF_CENTROIDS_SHAPE = build_reference(
    es_days[es_days.index < DISCOVERY_END], features=SHAPE_FEATURES, k=N_SHAPES)
log.info('L1b: эталон %d форм сессии на discovery (<%s)', N_SHAPES, DISCOVERY_END)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · L2 — Expanding walk-forward → OOS-леджер
# ═══════════════════════════════════════════════════════════════════════
# По месяцам: fit regime+shape на [start…m-1] → assign m (только прошлое).
# next-day: дескрипторы дня t-1 → торгуем день t; таргет = open(t)→close(t).
# Весь период становится OOS по построению.
WARMUP_MONTHS = 24
WF_NINIT      = 4    # n_init в WF (refit каждый месяц) — компромисс скорость/стабильность

_months     = pd.period_range(es_days.index.min(), es_days.index.max(), freq='M')
_first_test = es_days.index.min() + pd.DateOffset(months=WARMUP_MONTHS)
_blocks = []
for per in _months:
    m0 = per.to_timestamp(); m1 = (per + 1).to_timestamp()
    if m0 < _first_test:
        continue
    train = es_days[es_days.index < m0]
    test  = es_days[(es_days.index >= m0) & (es_days.index < m1)]
    if len(test) == 0 or len(train) < 252:
        continue
    # regime
    sc_r, gmm_r = fit_regime_model(train, REGIME_FEATURES, N_REGIMES, n_init=WF_NINIT)
    reg, regc = assign_regimes(test, sc_r, gmm_r,
                    canonical_map(sc_r, gmm_r, REF_CENTROIDS), REGIME_FEATURES)
    # shape
    sc_s, gmm_s = fit_regime_model(train, SHAPE_FEATURES, N_SHAPES, n_init=WF_NINIT)
    shp, shpc = assign_regimes(test, sc_s, gmm_s,
                    canonical_map(sc_s, gmm_s, REF_CENTROIDS_SHAPE), SHAPE_FEATURES)
    _blocks.append(pd.DataFrame(
        {'regime': reg, 'regime_conf': regc, 'shape': shp, 'shape_conf': shpc},
        index=test.index))

wf = pd.concat(_blocks).sort_index()
wf = wf.join(es_days[['open_px', 'close_px', 'open_location', 'direction']])

# next-day лаг: торгуем день t по дескрипторам завершённого дня t-1
wf['regime_prev']      = wf['regime'].shift(1)
wf['shape_prev']       = wf['shape'].shift(1)
wf['regime_conf_prev'] = wf['regime_conf'].shift(1)
wf['shape_conf_prev']  = wf['shape_conf'].shift(1)
# таргет дня t (open→close); open_location(t) — open-known, остаётся текущим
wf['fwd_ret'] = (wf['close_px'] - wf['open_px']) / wf['open_px']
wf['win']     = wf['fwd_ret'] > 0
wf = wf.dropna(subset=['regime_prev', 'shape_prev', 'fwd_ret'])

log.info('L2 OOS-леджер: %d дней  %s … %s', len(wf),
         wf.index.min().date(), wf.index.max().date())
log.info('Режимы (prev) распределение:\n%s',
         wf['regime_prev'].astype(int).value_counts().sort_index().to_string())
log.info('Формы (prev) распределение:\n%s',
         wf['shape_prev'].astype(int).value_counts().sort_index().to_string())
log.info('Overall: WR=%.1f%%  avg_ret=%.3f%%  n=%d',
         wf['win'].mean()*100, wf['fwd_ret'].mean()*100, len(wf))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · L3 — Оценка эджа (выборка-первична)
# ═══════════════════════════════════════════════════════════════════════
# Три теста ячейки ПРОТИВ остального пула, FDR по каждому:
#   p_t  — Welch t-test (среднее; чувствителен к хвостам)
#   p_mw — Mann-Whitney (ранговый; робастен к хвостам → больше мощности)
#   p_wr — two-proportion z (направленный WR vs baseline)
# Ранжируем по p_mw. min_n отсекает микровыборки.
from scipy import stats
from scipy.stats import norm

def wilson_lb(k, n, z=1.96):
    if n == 0: return 0.0
    p = k / n
    denom = 1 + z*z/n
    centre = p + z*z/(2*n)
    margin = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (centre - margin) / denom

def bh_fdr(pvals):
    p = np.asarray(pvals, float); n = len(p)
    order = np.argsort(p); q = np.empty(n); prev = 1.0
    for rank, idx in enumerate(order[::-1]):
        r = n - rank
        prev = min(prev, p[idx] * n / r); q[idx] = prev
    return q

def two_prop_p(k1, n1, k2, n2):
    if n1 == 0 or n2 == 0: return 1.0
    pp = (k1 + k2) / (n1 + n2)
    se = np.sqrt(pp*(1-pp)*(1/n1 + 1/n2))
    if se == 0: return 1.0
    return 2 * (1 - norm.cdf(abs((k1/n1 - k2/n2) / se)))

MIN_N = 40
Q_THRESH = 0.10
POOL_MEAN = wf['fwd_ret'].mean()
POOL_WR   = wf['win'].mean()
POOL_WINS = int(wf['win'].sum())
POOL_N    = len(wf)

def _cell_rows(by):
    rows = []
    for key, g in wf.groupby(by):
        n = len(g)
        if n < MIN_N: continue
        k    = int(g['win'].sum())
        rest = wf['fwd_ret'].drop(g.index)
        _, p_t  = stats.ttest_ind(g['fwd_ret'], rest, equal_var=False)
        _, p_mw = stats.mannwhitneyu(g['fwd_ret'], rest, alternative='two-sided')
        p_wr    = two_prop_p(k, n, POOL_WINS - k, POOL_N - n)
        rows.append({
            'level':    '×'.join(by),
            'cell':     key if isinstance(key, tuple) else (key,),
            'n':        n,
            'wr':       round(g['win'].mean()*100, 1),
            'wr_lb':    round(wilson_lb(k, n)*100, 1),
            'ret':      round(g['fwd_ret'].mean()*100, 3),
            'lift_ret': round((g['fwd_ret'].mean()-POOL_MEAN)*100, 3),
            'p_t':      p_t, 'p_mw': p_mw, 'p_wr': p_wr,
        })
    return rows

LEVELS = [
    ['regime_prev'], ['shape_prev'], ['open_location'],
    ['regime_prev','shape_prev'], ['regime_prev','open_location'],
    ['shape_prev','open_location'], ['regime_prev','shape_prev','open_location'],
]
allrows = []
for by in LEVELS:
    allrows += _cell_rows(by)
edge = pd.DataFrame(allrows)
for col in ['p_t', 'p_mw', 'p_wr']:
    edge['q' + col[1:]] = bh_fdr(edge[col].values)
edge['sig']  = (edge[['q_t','q_mw','q_wr']] < Q_THRESH).any(axis=1)
edge['side'] = np.where(edge['lift_ret'] >= 0, 'long', 'short')
edge['p']    = edge['p_mw']   # алиас для диагностики
edge = edge.sort_values('p_mw').reset_index(drop=True)

log.info('L3: baseline WR=%.1f%%  mean_ret=%.3f%%  (ячеек %d, min_n=%d)',
         POOL_WR*100, POOL_MEAN*100, len(edge), MIN_N)
log.info('L3 значимых q<%.2f:  t=%d  MW=%d  WR=%d',
         Q_THRESH, int((edge['q_t']<Q_THRESH).sum()),
         int((edge['q_mw']<Q_THRESH).sum()), int((edge['q_wr']<Q_THRESH).sum()))
log.info('Топ-15 по p_mw:\n%s',
         edge[['level','cell','n','wr','wr_lb','lift_ret','p_t','p_mw','p_wr','q_mw','side']]
             .head(15).to_string(index=False))

In [ ]:
edge.sort_values('p').head(15)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · L3-vol — Магнитудный таргет |fwd_ret| (волатильность)
# ═══════════════════════════════════════════════════════════════════════
# Волатильность кластеризуется и автокоррелирована → предсказуемость выше,
# чем у направления. Монетизация: сайзинг, "торговать при экспансии", straddle.
# Тест: Mann-Whitney на |fwd_ret| ячейки vs пул (робастно), + t-test.
wf['abs_ret'] = wf['fwd_ret'].abs()
POOL_ABS = wf['abs_ret'].mean()

def _vol_rows(by):
    rows = []
    for key, g in wf.groupby(by):
        n = len(g)
        if n < MIN_N: continue
        rest = wf['abs_ret'].drop(g.index)
        _, p_t  = stats.ttest_ind(g['abs_ret'], rest, equal_var=False)
        _, p_mw = stats.mannwhitneyu(g['abs_ret'], rest, alternative='two-sided')
        rows.append({
            'level':   '×'.join(by),
            'cell':    key if isinstance(key, tuple) else (key,),
            'n':       n,
            'abs_ret': round(g['abs_ret'].mean()*100, 3),
            'lift':    round((g['abs_ret'].mean()-POOL_ABS)*100, 3),
            'p_mw':    p_mw, 'p_t': p_t,
        })
    return rows

volrows = []
for by in LEVELS:
    volrows += _vol_rows(by)
vol = pd.DataFrame(volrows)
vol['q_mw'] = bh_fdr(vol['p_mw'].values)
vol['q_t']  = bh_fdr(vol['p_t'].values)
vol['sig']  = vol['q_mw'] < Q_THRESH
vol = vol.sort_values('p_mw').reset_index(drop=True)

log.info('L3-vol: baseline |ret|=%.3f%%  значимых q_mw<%.2f: %d из %d',
         POOL_ABS*100, Q_THRESH, int(vol['sig'].sum()), len(vol))
log.info('Топ-20 по магнитуде (p_mw):\n%s',
         vol[['level','cell','n','abs_ret','lift','p_mw','q_mw']].head(20).to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · L3-inc — Инкрементальность: нужен ли Далтон (MP)?
# ═══════════════════════════════════════════════════════════════════════
# Вложенные WF-регрессии предсказания vol |fwd_ret(t)| (Ridge, expanding):
#   M0 = наивная vol-персистентность (EWMA|ret|)
#   M1 = M0 + не-MP блок (intermarket / macro / vol)
#   M2 = M1 + MP/Dalton блок (IB/profile/POC/bimodality/open_location)
# ΔR²(M1-M0)=вклад intermarket/macro;  ΔR²(M2-M1)=ЧИСТЫЙ вклад Далтона.
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

MP_FEATURES = ['ib_norm','prof_norm','ib_profile_ratio','poc_bias','open_poc_dist',
               'x1_ibw','x2_prw','poc_position','bimodality','session_thrust']
NONMP_FEATURES = ['vix_percentile_252d','vix_change_5d','vol_ratio_5_20',
                  'yield_slope','yield_slope_change_10d','zn_mom_10d','cl_mom_20d','dx_mom_20d',
                  'er_20d','autocorr_lag1_20d','cl_es_corr_20d','zn_es_corr_20d',
                  'ret_skew_20d','range_pct_20d']

y     = wf['abs_ret']
naive = wf['abs_ret'].ewm(halflife=10).mean().shift(1).rename('ewma_abs').to_frame()
mp    = es_days[MP_FEATURES].shift(1).reindex(wf.index)              # завершённый день t-1
nmp   = es_days[NONMP_FEATURES].shift(1).reindex(wf.index)
oloc  = pd.get_dummies(wf['open_location'], prefix='ol').astype(float)  # open-known t (MP-контекст)

MODELS = {
    'M0_naive':            [naive],
    'M1_+intermkt_macro':  [naive, nmp],
    'M2_+MP_Dalton':       [naive, nmp, mp, oloc],
}
Xs = {k: pd.concat(v, axis=1) for k, v in MODELS.items()}

valid = y.dropna().index
for X in Xs.values():
    valid = valid.intersection(X.dropna().index)
_months = pd.period_range(valid.min(), valid.max(), freq='M')

def _wf_oos(X):
    Xv, yv = X.loc[valid], y.loc[valid]
    pred = pd.Series(np.nan, index=valid)
    for per in _months:
        m0 = per.to_timestamp(); m1 = (per + 1).to_timestamp()
        tr = Xv.index[Xv.index < m0]
        te = Xv.index[(Xv.index >= m0) & (Xv.index < m1)]
        if len(tr) < 252 or len(te) == 0:
            continue
        sc = StandardScaler().fit(Xv.loc[tr])
        rg = Ridge(alpha=1.0).fit(sc.transform(Xv.loc[tr]), yv.loc[tr])
        pred.loc[te] = rg.predict(sc.transform(Xv.loc[te]))
    return pred

res = {}
for k, X in Xs.items():
    pr = _wf_oos(X); m = pr.dropna().index
    yt, pp = y.loc[m], pr.loc[m]
    sse = ((yt - pp)**2).sum(); sst = ((yt - yt.mean())**2).sum()
    res[k] = {'OOS_R2': 1 - sse/sst,
              'Spearman': spearmanr(pp, yt).correlation,
              'MAE_%': (yt - pp).abs().mean()*100, 'n': len(m)}

rdf = pd.DataFrame(res).T
log.info('L3-inc · OOS-предсказание vol |fwd_ret|:\n%s', rdf.round(4).to_string())
d_macro = res['M1_+intermkt_macro']['OOS_R2'] - res['M0_naive']['OOS_R2']
d_mp    = res['M2_+MP_Dalton']['OOS_R2']      - res['M1_+intermkt_macro']['OOS_R2']
log.info('ΔR² intermarket/macro (M1-M0): %+.4f', d_macro)
log.info('ΔR² MP/DALTON       (M2-M1): %+.4f   <-- решение: держать теорию или дропнуть', d_mp)
log.info('Δ Spearman MP/Dalton: %+.4f',
         res['M2_+MP_Dalton']['Spearman'] - res['M1_+intermkt_macro']['Spearman'])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · #4a — Intraday post-IB леджер (домашнее поле Далтона)
# ═══════════════════════════════════════════════════════════════════════
# Вход ПОСЛЕ IB (60 мин) дня t → exit на close t. Все фичи известны на закрытии
# IB (open-known): структура IB + open_location + activity/open_type из classify
# (они считаются по первым 60 мин → causal). Таргет = post-IB ход.
_es = es_rth.copy()
_es['_d'] = _es.index.normalize()
ib_end = _es.groupby('_d').apply(lambda g: g['open'].iloc[60] if len(g) > 60 else np.nan)
es_days['ib_end_px']      = ib_end.reindex(es_days.index)
es_days['post_ib_ret']    = (es_days['close_px'] - es_days['ib_end_px']) / es_days['ib_end_px']
es_days['open_pos_in_ib'] = ((es_days['open_px'] - es_days['ib_low']) /
                             (es_days['ib_high'] - es_days['ib_low']).replace(0, np.nan)).clip(0, 1)
es_days['ib_thrust']      = (es_days['ib_end_px'] - es_days['open_px']) / (es_days['irv'] * es_days['open_px'])

_pi = es_days['post_ib_ret'].dropna()
log.info('#4a post-IB: %d дней  baseline WR=%.1f%%  avg=%.3f%%  |move|=%.3f%%',
         len(_pi), (_pi > 0).mean()*100, _pi.mean()*100, _pi.abs().mean()*100)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# НОВАЯ АРХИТЕКТУРА · #4b — РЕШАЮЩИЙ тест Далтона (intraday post-IB)
# ═══════════════════════════════════════════════════════════════════════
# (1) Инкрементальность MP vs non-MP для post-IB НАПРАВЛЕНИЯ и VOL (Ridge WF).
# (2) Edge-скан направления по MP-факторам (open_location/activity/open_type), FDR.
# MP-блок известен на закрытии IB (causal); non-MP лагируется на t-1.

# ── блоки фич ──
mp_cont = es_days[['ib_norm', 'open_pos_in_ib', 'ib_thrust']]
mp_dum  = pd.concat([pd.get_dummies(es_days['open_location'], prefix='ol'),
                     pd.get_dummies(es_days['activity'],      prefix='act'),
                     pd.get_dummies(es_days['open_type'],     prefix='ot')], axis=1).astype(float)
MP_BLK  = pd.concat([mp_cont, mp_dum], axis=1)
NMP_BLK = es_days[NONMP_FEATURES].shift(1)   # intermarket/macro дня t-1

def _nested_oos(blocks, y):
    idx = y.dropna().index
    for _, X in blocks: idx = idx.intersection(X.dropna().index)
    months = pd.period_range(idx.min(), idx.max(), freq='M')
    out = {}
    for name, X in blocks:
        Xv, yv = X.loc[idx], y.loc[idx]
        pred = pd.Series(np.nan, index=idx)
        for per in months:
            m0 = per.to_timestamp(); m1 = (per + 1).to_timestamp()
            tr = Xv.index[Xv.index < m0]; te = Xv.index[(Xv.index >= m0) & (Xv.index < m1)]
            if len(tr) < 252 or len(te) == 0: continue
            sc = StandardScaler().fit(Xv.loc[tr])
            rg = Ridge(alpha=1.0).fit(sc.transform(Xv.loc[tr]), yv.loc[tr])
            pred.loc[te] = rg.predict(sc.transform(Xv.loc[te]))
        m = pred.dropna().index; yt, pp = yv.loc[m], pred.loc[m]
        sse = ((yt - pp)**2).sum(); sst = ((yt - yt.mean())**2).sum()
        out[name] = {'R2': 1 - sse/sst, 'Spearman': spearmanr(pp, yt).correlation, 'n': len(m)}
    return out

# (1a) НАПРАВЛЕНИЕ
y_dir   = es_days['post_ib_ret']
nv_dir  = y_dir.shift(1).rename('mom').to_frame()
rd = _nested_oos([('M0_naive', nv_dir),
                  ('M1_+nonMP', pd.concat([nv_dir, NMP_BLK], axis=1)),
                  ('M2_+MP_Dalton', pd.concat([nv_dir, NMP_BLK, MP_BLK], axis=1))], y_dir)
# (1b) VOL
y_vol   = es_days['post_ib_ret'].abs()
nv_vol  = y_vol.ewm(halflife=10).mean().shift(1).rename('ewma').to_frame()
rv = _nested_oos([('M0_naive', nv_vol),
                  ('M1_+nonMP', pd.concat([nv_vol, NMP_BLK], axis=1)),
                  ('M2_+MP_Dalton', pd.concat([nv_vol, NMP_BLK, MP_BLK], axis=1))], y_vol)

log.info('#4b НАПРАВЛЕНИЕ post-IB:\n%s', pd.DataFrame(rd).T.round(4).to_string())
log.info('  ΔR² MP/Dalton (направление): %+.4f', rd['M2_+MP_Dalton']['R2'] - rd['M1_+nonMP']['R2'])
log.info('#4b VOL post-IB:\n%s', pd.DataFrame(rv).T.round(4).to_string())
log.info('  ΔR² MP/Dalton (vol): %+.4f', rv['M2_+MP_Dalton']['R2'] - rv['M1_+nonMP']['R2'])

# (2) Edge-скан направления по MP-факторам
pir = es_days.dropna(subset=['post_ib_ret']).copy()
pir['w'] = pir['post_ib_ret'] > 0
base_wr  = pir['w'].mean(); tot_w = int(pir['w'].sum()); tot_n = len(pir)
rows = []
for col in ['open_location', 'activity', 'open_type']:
    for k, g in pir.groupby(col):
        n = len(g)
        if n < MIN_N: continue
        kk = int(g['w'].sum())
        rows.append({'factor': col, 'cell': k, 'n': n,
                     'wr': round(g['w'].mean()*100, 1),
                     'wr_lb': round(wilson_lb(kk, n)*100, 1),
                     'ret': round(g['post_ib_ret'].mean()*100, 3),
                     'p': two_prop_p(kk, n, tot_w - kk, tot_n - n)})
ed = pd.DataFrame(rows)
ed['q'] = bh_fdr(ed['p'].values)
ed['sig'] = ed['q'] < Q_THRESH
ed = ed.sort_values('p').reset_index(drop=True)
log.info('post-IB direction edge по MP-факторам (baseline WR=%.1f%%), значимых q<%.2f: %d',
         base_wr*100, Q_THRESH, int(ed['sig'].sum()))
log.info('\n%s', ed.head(15).to_string(index=False))